In [1]:
!pip uninstall numpy -y
!pip install numpy==1.26.4

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
  Using cached numpy-1.26.4-cp313-cp313-macosx_10_16_x86_64.whl
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [2]:
!pip install dlib

In [3]:
!pip install firebase-admin


In [ ]:
import cv2
import dlib
import numpy as np
from scipy.spatial import distance
import firebase_admin
from firebase_admin import credentials, db
import time

# FIREBASE INITIALIZATION

cred = credentials.Certificate("serviceAccountKey.json")

# Initialize Firebase app
firebase_admin.initialize_app(cred, {
    "databaseURL": "https://driver-drowsiness-system-d98d9-default-rtdb.firebaseio.com/"
})


events_ref = db.reference("drowsiness_events")


# FUNCTION to send event to Firebase
def send_firebase_event(trigger_type):
    event_data = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "event": "DROWSINESS DETECTED",
        "trigger": trigger_type
    }
    events_ref.push(event_data)
    print("Firebase event uploaded:", event_data)



# EAR FUNCTION
def eye_aspect_ratio(eye):
    A = distance.euclidean(eye[1], eye[5])
    B = distance.euclidean(eye[2], eye[4])
    C = distance.euclidean(eye[0], eye[3])
    ear = (A + B) / (2.0 * C)
    return ear


# DLIB SETUP
detector = dlib.get_frontal_face_detector()
predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")

LEFT_EYE = list(range(36, 42))
RIGHT_EYE = list(range(42, 48))
CHIN = 8
NOSE = 33
HEAD_DROP_THRESHOLD = 90
HEAD_NOD_FRAMES = 45



EAR_THRESHOLD = 0.25
CONSEC_FRAMES = 60
blink_counter = 0
head_counter = 0
sent_event = False   # prevents spamming Firebase



# VIDEO STREAM
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = detector(gray)

    for face in faces:
        shape = predictor(gray, face)
        landmarks = np.array([[p.x, p.y] for p in shape.parts()])
        #EAR Detection

        left_eye = landmarks[LEFT_EYE]
        right_eye = landmarks[RIGHT_EYE]

        left_ear = eye_aspect_ratio(left_eye)
        right_ear = eye_aspect_ratio(right_eye)
        ear = (left_ear + right_ear) / 2.0

        # Draw eye points
        for (x, y) in np.concatenate((left_eye, right_eye)):
            cv2.circle(frame, (x, y), 2, (0, 255, 0), -1)

        #EAR Logic
        if ear < EAR_THRESHOLD:
            blink_counter += 1
        else:
            blink_counter = 0

        # HEAD NODDING DETECTION
        nose_y = landmarks[NOSE][1]
        chin_y = landmarks[CHIN][1]
        head_drop = chin_y - nose_y


        cv2.putText(frame, f"EAR: {ear:.2f}", (30, 90),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
        cv2.putText(frame, f"Head Drop: {int(head_drop)}", (30, 120),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)

        if head_drop < HEAD_DROP_THRESHOLD:
            head_counter += 1
        else:
            head_counter = 0


        # DROWSINESS DETECTION
        if blink_counter >= CONSEC_FRAMES or head_counter >= HEAD_NOD_FRAMES:
            cv2.putText(frame, "DROWSY!", (50, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

            trigger = "EAR" if blink_counter >= CONSEC_FRAMES else "HEAD_NOD"
            print(f"DROWSY! [{trigger}]")

            if not sent_event:
                send_firebase_event(trigger)
                sent_event = True
        else:
            sent_event = False

    cv2.imshow("Drowsiness Detection (EAR + Head Nodding)", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

# CLEANUP
cap.release()
cv2.destroyAllWindows()


DROWSY! [HEAD_NOD]
Firebase event uploaded: {'timestamp': '2026-02-06 13:50:45', 'event': 'DROWSINESS DETECTED', 'trigger': 'HEAD_NOD'}
DROWSY! [HEAD_NOD]
DROWSY! [HEAD_NOD]
DROWSY! [HEAD_NOD]
DROWSY! [HEAD_NOD]
DROWSY! [HEAD_NOD]
DROWSY! [HEAD_NOD]
DROWSY! [EAR]
Firebase event uploaded: {'timestamp': '2026-02-06 13:51:48', 'event': 'DROWSINESS DETECTED', 'trigger': 'EAR'}
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
DROWSY! [EAR]
